In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch01-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "01-04-mol-and-sdf.py"

# 把三種結構和水溶解度一起交給同事

第一次執行會下載 ESOL。先看第一筆 SDF 文字，再對照圖中的三個實際讀回的結構。要做練習，可把程式加在分析格末尾，重新執行分析格與最後一格。

In [ ]:
%%writefile "{script}"
from io import BytesIO, StringIO
from pathlib import Path
import os
import deepchem as dc
from rdkit import Chem
from rdkit.Chem import Draw, rdDepictor

# 只用三筆 ESOL 量測，標籤保持原始 log10(mol/L) 尺度。
_, (data,), _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None,
    transformers=[], reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
names = {"Oc1ccccc1": "Phenol", "COc1ccccc1": "Anisole", "Cc1ccc(O)cc1": "p-Cresol"}
molecules = []
for i, (smiles, value, weight) in enumerate(zip(data.ids, data.y[:, 0], data.w[:, 0])):
    mol = Chem.MolFromSmiles(smiles)
    key = Chem.MolToSmiles(mol)
    if key in names and weight > 0:
        rdDepictor.Compute2DCoords(mol)
        mol.SetProp("_Name", names[key])
        mol.SetProp("RecordID", f"ESOL-row-{i + 1}")
        mol.SetProp("logS", str(float(value)))
        mol.SetProp("Endpoint", "log10(S / (mol/L))")
        molecules.append(mol)
assert len(molecules) == len({m.GetProp("_Name") for m in molecules}) == 3

def as_sdf(mols):
    buffer = StringIO()
    with Chem.SDWriter(buffer) as writer:
        writer.SetProps(["RecordID", "logS", "Endpoint"])
        for mol in mols:
            writer.write(mol)
    return buffer.getvalue()

def read_sdf(text):
    return list(Chem.ForwardSDMolSupplier(BytesIO(text.encode("utf-8"))))

sdf = as_sdf(molecules)
restored = read_sdf(sdf)
assert len(restored) == len(molecules) and all(m is not None for m in restored)
for original, mol in zip(molecules, restored):
    assert Chem.MolToSmiles(original) == Chem.MolToSmiles(mol)
    for field in ["_Name", "RecordID", "logS", "Endpoint"]:
        assert original.GetProp(field) == mol.GetProp(field)
    assert not mol.GetConformer().Is3D()
    print(mol.GetProp("_Name"), mol.GetProp("RecordID"),
          mol.GetNumAtoms(), mol.GetNumBonds(), mol.GetProp("logS"))
print("record separators:", sdf.splitlines().count("$$$$"))
print(sdf.split("$$$$")[0])
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
legends = [f"{m.GetProp('_Name')}   |   {m.GetProp('RecordID')}\nlog S = {float(m.GetProp('logS')):.2f}"
           for m in restored]
figure = Draw.MolsToGridImage(restored, molsPerRow=3, subImgSize=(320, 230), legends=legends)
figure.save(out / "01-04-mol-and-sdf.png")


In [ ]:
from IPython.display import Image, display

run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.removeprefix("This module requires PyTorch to be installed.\n").strip())
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "01-04-mol-and-sdf.png")))